In [1]:
!pip install kagglehub -q

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import os
import gc
import joblib
import kagglehub

from sklearn.preprocessing import StandardScaler

In [4]:
BASE_PATH = "/content/drive/MyDrive/Zero-Day-IoT-Project"

DATA_PATH = BASE_PATH + "/01_Data"
MODEL_PATH = BASE_PATH + "/03_Models"

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)

print("Folders ready")

Folders ready


In [5]:
dataset_path = kagglehub.dataset_download("sibasispradhan/edge-iiotset-dataset")

print("Downloaded to:", dataset_path)

100%|██████████| 351M/351M [00:01<00:00, 210MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/datasets/sibasispradhan/edge-iiotset-dataset/versions/2


In [6]:
os.listdir(dataset_path)

['ML-EdgeIIoT-dataset.csv',
 'DNN-EdgeIIoT-dataset.csv',
 'live_data_training.csv']

In [20]:
# Load Main CSV
file_path = dataset_path + "/ML-EdgeIIoT-dataset.csv"

df = pd.read_csv(file_path, low_memory=False)

print("Original Shape:", df.shape)
df.head()

Original Shape: (157800, 63)


,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
1,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
2,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
3,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
4,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM


In [8]:
# Drop Irrelevant / Heavy Columns
drop_cols = [
    'frame.time',
    'ip.src_host',
    'ip.dst_host',
    'arp.src.proto_ipv4',
    'arp.dst.proto_ipv4',
    'http.request.full_uri',
    'http.request.uri.query',
    'http.file_data',
    'icmp.transmit_timestamp',
    'mqtt.msg'
]

df.drop(columns=drop_cols, inplace=True, errors='ignore')

print("After Drop:", df.shape)

After Drop: (157800, 53)


In [9]:
# Remove Nulls + Duplicates
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

print("After Cleaning:", df.shape)

After Cleaning: (152389, 53)


In [10]:
# Check Labels
print("Attack Label Distribution:\n")
print(df['Attack_label'].value_counts())

print("\nAttack Type Distribution:\n")
print(df['Attack_type'].value_counts())

Attack Label Distribution:

Attack_label
1    128237
0     24152
Name: count, dtype: int64

Attack Type Distribution:

Attack_type
Normal                   24152
DDoS_UDP                 14498
DDoS_ICMP                13096
DDoS_HTTP                10559
SQL_injection            10286
DDoS_TCP                 10247
Uploading                10237
Vulnerability_scanner    10062
Password                  9978
Backdoor                  9866
Ransomware                9690
XSS                       9550
Port_Scanning             8921
Fingerprinting             853
MITM                       394
Name: count, dtype: int64


In [11]:
# Save Attack Type Mapping
mapping = pd.DataFrame({
    "Attack_type": sorted(df["Attack_type"].unique())
})

mapping.to_csv(DATA_PATH + "/attack_type_mapping.csv", index=False)

print("Mapping Saved")
mapping.head(15)

Mapping Saved


,Attack_type
0,Backdoor
1,DDoS_HTTP
2,DDoS_ICMP
3,DDoS_TCP
4,DDoS_UDP
5,Fingerprinting
6,MITM
7,Normal
8,Password
9,Port_Scanning


In [12]:
# Encode Text Columns
obj_cols = df.select_dtypes(include='object').columns.tolist()

print("Object Columns:", obj_cols)

for col in obj_cols:
    df[col] = df[col].astype('category').cat.codes

print("Encoding Completed")

Object Columns: ['http.request.method', 'http.referer', 'http.request.version', 'tcp.options', 'tcp.payload', 'tcp.srcport', 'dns.qry.name.len', 'mqtt.conack.flags', 'mqtt.protoname', 'mqtt.topic', 'Attack_type']
Encoding Completed


In [13]:
# Optimize Memory Types
for col in df.select_dtypes(include='float64').columns:
    df[col] = df[col].astype('float32')

for col in df.select_dtypes(include='int64').columns:
    df[col] = df[col].astype('int32')

print(df.dtypes.value_counts())

float32    41
int8        8
int32       2
int16       2
Name: count, dtype: int64


In [14]:
# Define Known vs Zero-Day Classes
# Training classes
known_classes = [7,4,2,1,11,3,8,0,10,9]

# Hidden zero-day classes
unknown_classes = [12,13,14,5,6]

known_df = df[df['Attack_type'].isin(known_classes)].copy()
zero_df  = df[df['Attack_type'].isin(unknown_classes)].copy()

print("Known Train Shape:", known_df.shape)
print("Zero-Day Test Shape:", zero_df.shape)

Known Train Shape: (121293, 53)
Zero-Day Test Shape: (31096, 53)


In [15]:
# Save Unscaled Datasets
df.to_parquet(DATA_PATH + "/master_clean.parquet")
known_df.to_parquet(DATA_PATH + "/known_train.parquet")
zero_df.to_parquet(DATA_PATH + "/zero_day_test.parquet")

print("Unscaled datasets saved")

Unscaled datasets saved


In [16]:
# Create Scaler for Autoencoder
feature_cols = [c for c in df.columns if c not in ['Attack_label', 'Attack_type']]

scaler = StandardScaler()

# Fit only on known normal traffic rows
normal_known = known_df[known_df['Attack_label'] == 0]

scaler.fit(normal_known[feature_cols])

joblib.dump(scaler, MODEL_PATH + "/scaler.pkl")

print("Scaler Saved")

Scaler Saved


In [17]:
print("FINAL OUTPUTS CREATED")
print("----------------------")
print("master_clean.parquet")
print("known_train.parquet")
print("zero_day_test.parquet")
print("attack_type_mapping.csv")
print("scaler.pkl")

FINAL OUTPUTS CREATED
----------------------
master_clean.parquet
known_train.parquet
zero_day_test.parquet
attack_type_mapping.csv
scaler.pkl


In [18]:
# Free RAM
del df
del known_df
del zero_df

gc.collect()

print("Memory Cleared")

Memory Cleared
